# 🎙️ Entraînement & Quantification INT4 / INT8 — Reverb-TTS (Français)

Ce notebook fine-tune **`software-mansion/Reverb-TTS`** (297M paramètres, issu d'OmniVoice élagué et distillé) sur votre dataset prêt **`dataset_reverb_narrateur.zip`**, puis exporte **les deux versions quantifiées** prêtes pour l'inférence :
- **INT4** (`group_size=32`, ~191 Mo — format principal ultra-rapide)
- **INT8** (`per-channel`, ~307 Mo — format alternatif sans perte)

### Caractéristiques
- **Zéro modification du dataset** : charge directement `dataset_reverb_narrateur.zip`.
- **Auto-adaptation GPU** : ajuste automatiquement `BATCH_SIZE`, `GRAD_ACCUM` et `DTYPE` selon la VRAM (A100, L4, T4). *(Les TPU ne sont pas compatibles avec le cache dynamique et les longueurs variables de Reverb-TTS.)*
- **Historique toutes les 20 minutes** : sauvegarde un snapshot complet dans `Mon Drive/reverb_tts/checkpoints_history/epoch_XXXX/` toutes les 20 minutes pour pouvoir tester différentes époques, plus `last/` pour reprise automatique.

## 1. Configuration & Détection GPU

In [ ]:
import os, torch
from google.colab import drive

assert torch.cuda.is_available(), (
    "❌ Aucun GPU NVIDIA (CUDA) détecté ! "
    "Reverb-TTS (DynamicCache + longueurs dynamiques) nécessite un GPU CUDA (T4, L4 ou A100), pas un TPU."
)

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# === PARAMÈTRES GÉNÉRAUX ===
BASE_MODEL = "software-mansion/Reverb-TTS"
DATASET_ZIP = "/content/drive/MyDrive/reverb_tts/dataset_reverb_narrateur.zip"
DRIVE_OUT_DIR = "/content/drive/MyDrive/reverb_tts/narrateur_model"
WORK_DIR = "/content/work_reverb_train"
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

# Hyperparamètres LoRA (recommandés pour Reverb-TTS 297M)
MAX_EPOCHS = 15
LR = 1e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
SAVE_EVERY_MINUTES = 20  # Sauvegarde historique sur Drive toutes les 20 minutes
AUTO_DISCONNECT = False  # Passer à True pour couper automatiquement le GPU à la fin

# === AUTO-CONFIGURATION SELON LE GPU ===
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
major_cc, _ = torch.cuda.get_device_capability(0)
cpu_cores = os.cpu_count() or 2

if vram_gb >= 38:         # A100 (40/80 Go), H100
    BATCH_SIZE = 8
    GRAD_ACCUM = 2        # batch effectif = 16
    DTYPE = torch.bfloat16
    NUM_WORKERS = min(4, cpu_cores)
elif vram_gb >= 20:       # L4 (24 Go)
    BATCH_SIZE = 4
    GRAD_ACCUM = 4
    DTYPE = torch.bfloat16 if major_cc >= 8 else torch.float16
    NUM_WORKERS = min(4, cpu_cores)
elif vram_gb >= 14:       # T4 (16 Go), V100
    BATCH_SIZE = 2
    GRAD_ACCUM = 8
    DTYPE = torch.float16
    NUM_WORKERS = min(2, cpu_cores)
else:
    BATCH_SIZE = 1
    GRAD_ACCUM = 16
    DTYPE = torch.float16
    NUM_WORKERS = min(2, cpu_cores)

print(f"🖥️ GPU : {gpu_name} ({vram_gb:.1f} Go VRAM, CC {major_cc}.x)")
print(f"⚙️ Config auto : BATCH={BATCH_SIZE} x ACCUM={GRAD_ACCUM} (eff={BATCH_SIZE*GRAD_ACCUM}) | DTYPE={DTYPE}")
assert os.path.exists(DATASET_ZIP), f"Dataset introuvable : {DATASET_ZIP}"

## 2. Installation de `reverb-tts`, `peft` et `safetensors`

In [ ]:
!pip uninstall -y -q torchao
!pip install -q reverb-tts peft safetensors huggingface_hub soundfile tqdm


## 3. Chargement du Dataset ZIP & Encodage des Tokens Audio (24 kHz → Higgs Audio v2)

In [ ]:
import json, zipfile
import soundfile as sf
from tqdm import tqdm
from reverb_tts import TTS
from reverb_tts.tts import _prompt

DATA_DIR = os.path.join(WORK_DIR, "dataset")
CACHE_PT = os.path.join(WORK_DIR, "encoded_dataset.pt")

if not os.path.exists(DATA_DIR):
    with zipfile.ZipFile(DATASET_ZIP, "r") as z:
        z.extractall(DATA_DIR)

# Chargement du modèle de base en pleine précision (fp32/fp16) pour l'entraînement
tts = TTS.from_pretrained(BASE_MODEL, variant="fp16", device="cuda")
tokenizer = tts.tokenizer
codec = tts.codec

def load_jsonl(name):
    items = []
    for root, _, files in os.walk(DATA_DIR):
        if name in files:
            with open(os.path.join(root, name), "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        items.append((root, json.loads(line)))
    return items

if os.path.exists(CACHE_PT):
    print(f"⚡ Chargement du cache encodé : {CACHE_PT}")
    cached = torch.load(CACHE_PT, weights_only=False)
    train_samples, dev_samples = cached["train"], cached["dev"]
else:
    def encode_split(split_name):
        raw = load_jsonl(f"{split_name}.jsonl")
        encoded = []
        with torch.no_grad():
            for base_dir, r in tqdm(raw, desc=f"Encodage {split_name}"):
                wav_path = os.path.join(base_dir, r["audio_path"])
                if not os.path.exists(wav_path):
                    continue
                wav, sr = sf.read(wav_path, dtype="float32")
                if wav.ndim > 1:
                    wav = wav.mean(axis=1)
                wav_t = torch.from_numpy(wav)[None, None].to("cuda")
                codes = codec.encode(wav_t).audio_codes[0].cpu().to(torch.int16)  # [8, F]
                encoded.append({
                    "text": r["text"],
                    "codes": codes,
                    "language": r.get("language", "fr")
                })
        return encoded

    train_samples = encode_split("train")
    dev_samples = encode_split("dev")
    torch.save({"train": train_samples, "dev": dev_samples}, CACHE_PT)

print(f"✅ Dataset encodé : {len(train_samples)} train | {len(dev_samples)} dev")

## 4. Entraînement LoRA (Masked-Diffusion) + Historique toutes les 20 minutes

In [ ]:
import glob, math, random, shutil, time
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from peft import LoraConfig, get_peft_model, PeftModel
from reverb_tts.model import CODEBOOKS, CODES, MASK

CKPT_HISTORY_DIR = os.path.join(DRIVE_OUT_DIR, "checkpoints_history")
CKPT_LAST_DIR = os.path.join(DRIVE_OUT_DIR, "last")
os.makedirs(CKPT_HISTORY_DIR, exist_ok=True)
os.makedirs(CKPT_LAST_DIR, exist_ok=True)

class ReverbDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        # 70% avec un autre clip du narrateur en référence (renforce le clonage), 30% sans référence (voix ancrée)
        if len(self.samples) > 1 and random.random() < 0.7:
            ref = self.samples[random.randrange(len(self.samples))]
            ref_codes = ref["codes"].long()
            ref_text = ref["text"]
        else:
            ref_codes = torch.empty((CODEBOOKS, 0), dtype=torch.long)
            ref_text = ""
        prompt_str = _prompt(item["text"], ref_text, item["language"])
        text_ids = tokenizer(prompt_str, return_tensors="pt").input_ids[0]
        return text_ids, ref_codes, item["codes"].long()

def collate_batch(batch):
    return batch  # traitement séquence par séquence avec masquage dynamique

network = tts.network.to("cuda").to(DTYPE)
# Libérer le codec pendant l'entraînement pour économiser la VRAM
tts.codec.cpu()
torch.cuda.empty_cache()

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

start_epoch = 0
state_file = os.path.join(CKPT_LAST_DIR, "trainer_state.json")
if os.path.exists(os.path.join(CKPT_LAST_DIR, "adapter_config.json")):
    print(f"🔄 Reprise LoRA depuis : {CKPT_LAST_DIR}")
    network.llm = PeftModel.from_pretrained(network.llm, CKPT_LAST_DIR, is_trainable=True)
    if os.path.exists(state_file):
        start_epoch = json.load(open(state_file))["epoch"] + 1
else:
    network.llm = get_peft_model(network.llm, lora_cfg)

network.llm.print_trainable_parameters()
network.train()

optimizer = torch.optim.AdamW([p for p in network.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
loader = DataLoader(ReverbDataset(train_samples), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch, num_workers=0)

offsets = CODES * torch.arange(CODEBOOKS, device="cuda")[:, None]
head_weights = network.audio_heads.weight.view(CODEBOOKS, CODES, -1)

def compute_sample_loss(text_ids, ref_codes, target_codes):
    text_emb = network.llm.get_base_model().embed_tokens(text_ids.to("cuda"))
    if ref_codes.size(1) > 0:
        ref_emb = network.audio_embeddings(ref_codes.to("cuda") + offsets).sum(0)
        prefix_emb = torch.cat([text_emb, ref_emb], dim=0)
    else:
        prefix_emb = text_emb

    F_len = target_codes.size(1)
    tgt = target_codes.to("cuda")
    # Masquage diffusion (t entre 0.1 et 1.0)
    t = random.uniform(0.1, 1.0)
    mask_prob = 1.0 - math.cos(t * math.pi / 2.0)
    mask = (torch.rand((CODEBOOKS, F_len), device="cuda") < mask_prob)
    if not mask.any():
        mask[0, random.randrange(F_len)] = True

    masked_tgt = torch.where(mask, MASK, tgt)
    tgt_emb = network.audio_embeddings(masked_tgt + offsets).sum(0)
    full_emb = torch.cat([prefix_emb, tgt_emb], dim=0).unsqueeze(0).to(DTYPE)
    P = prefix_emb.size(0)
    pos_ids = torch.arange(P + F_len, device="cuda").unsqueeze(0)

    hidden = network.llm(inputs_embeds=full_emb, position_ids=pos_ids, use_cache=False).last_hidden_state[0, P:]  # [F, H]
    logits = torch.einsum("fh,cnh->cfn", hidden.float(), head_weights.float())  # [8, F, 1025]

    loss = F.cross_entropy(logits[mask], tgt[mask])
    return loss

def save_checkpoint(epoch, avg_loss, tag="last"):
    out_dir = CKPT_LAST_DIR if tag == "last" else os.path.join(CKPT_HISTORY_DIR, tag)
    os.makedirs(out_dir, exist_ok=True)
    network.llm.save_pretrained(out_dir)
    with open(os.path.join(out_dir, "trainer_state.json"), "w") as f:
        json.dump({"epoch": epoch, "loss": avg_loss, "timestamp": time.time()}, f)
    print(f"💾 Checkpoint sauvegardé : {out_dir} (epoch={epoch}, loss={avg_loss:.4f})")

last_history_save = time.time()
for epoch in range(start_epoch, MAX_EPOCHS):
    total_loss, steps = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for step_idx, batch in enumerate(loader):
        batch_loss = 0.0
        for text_ids, ref_codes, target_codes in batch:
            loss = compute_sample_loss(text_ids, ref_codes, target_codes) / (len(batch) * GRAD_ACCUM)
            loss.backward()
            batch_loss += float(loss.item()) * GRAD_ACCUM
        total_loss += batch_loss
        steps += 1

        if (step_idx + 1) % GRAD_ACCUM == 0 or (step_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        # Sauvegarde historique sur Drive toutes les 20 minutes
        if time.time() - last_history_save >= SAVE_EVERY_MINUTES * 60:
            cur_loss = total_loss / max(1, steps)
            save_checkpoint(epoch, cur_loss, tag=f"epoch_{epoch:04d}_loss_{cur_loss:.4f}")
            save_checkpoint(epoch, cur_loss, tag="last")
            last_history_save = time.time()

    avg_loss = total_loss / max(1, steps)
    print(f"Epoch {epoch+1}/{MAX_EPOCHS} | loss={avg_loss:.4f}")

# Sauvegarde finale
save_checkpoint(MAX_EPOCHS - 1, avg_loss, tag=f"epoch_{MAX_EPOCHS-1:04d}_final")
save_checkpoint(MAX_EPOCHS - 1, avg_loss, tag="last")

## 5. Fusion LoRA & Export Quantifié **INT4** + **INT8** + Test d'Écoute

Vous pouvez choisir n'importe quel checkpoint de `checkpoints_history/` ci-dessous (par défaut `last`) pour exporter les poids en **INT4** (`group_size=32`, ~191 Mo) et **INT8** (`per-channel`, ~307 Mo) au format natif `Reverb-TTS`.

In [ ]:
import json, shutil
import soundfile as sf
import IPython.display as ipd
from safetensors.torch import save_file
from huggingface_hub import snapshot_download
from reverb_tts.model import QUANTIZED, SCALE

# Choisir le checkpoint à exporter (ex: CKPT_LAST_DIR ou un dossier de checkpoints_history/)
CKPT_TO_EXPORT = CKPT_LAST_DIR

# Recharger un réseau propre fp16, appliquer et fusionner le LoRA
base_tts = TTS.from_pretrained(BASE_MODEL, variant="fp16", device="cpu")
peft_llm = PeftModel.from_pretrained(base_tts.network.llm, CKPT_TO_EXPORT)
base_tts.network.llm = peft_llm.merge_and_unload()
merged_state = {k: v.detach().cpu() for k, v in base_tts.network.state_dict().items()}

# Récupérer config.json et tokenizer* de référence
src_root = snapshot_download(BASE_MODEL, allow_patterns=["model/safetensors/fp16/config.json", "model/tokenizer*", "model/chat_template.jinja"])
base_cfg = json.load(open(os.path.join(src_root, "model/safetensors/fp16/config.json"), "r", encoding="utf-8"))

def quantize_and_save(variant_name, bits, group_size=None):
    out_dir = os.path.join(DRIVE_OUT_DIR, f"export_{variant_name}")
    model_dir = os.path.join(out_dir, "model")
    var_dir = os.path.join(model_dir, "safetensors", variant_name)
    os.makedirs(var_dir, exist_ok=True)

    # Copier les fichiers tokenizer
    for fn in os.listdir(os.path.join(src_root, "model")):
        src_f = os.path.join(src_root, "model", fn)
        if os.path.isfile(src_f):
            shutil.copy2(src_f, os.path.join(model_dir, fn))

    q_weights = {}
    for name, tensor in merged_state.items():
        if name.startswith(QUANTIZED) and tensor.ndim == 2 and name.endswith(".weight"):
            w = tensor.float()
            if bits == 4:
                assert w.size(1) % group_size == 0
                wg = w.unflatten(-1, (-1, group_size))
                scale = (wg.abs().amax(dim=-1) / 7.0).clamp(min=1e-8).to(torch.float16)
                q = torch.clamp(torch.round(wg / scale.float()[..., None]), -8, 7).to(torch.int16) + 8
                q_flat = q.flatten(-2)
                packed = (q_flat[:, 0::2] | (q_flat[:, 1::2] << 4)).to(torch.uint8)
                q_weights[name] = packed
                q_weights[f"{name}{SCALE}"] = scale
            else:  # int8 per output channel
                scale = (w.abs().amax(dim=-1, keepdim=True) / 127.0).clamp(min=1e-8).to(torch.float16)
                q = torch.clamp(torch.round(w / scale.float()), -127, 127).to(torch.int8)
                q_weights[name] = q
                q_weights[f"{name}{SCALE}"] = scale
        else:
            q_weights[name] = tensor.to(torch.float16) if tensor.is_floating_point() else tensor

    cfg = dict(base_cfg)
    cfg["quantization"] = {"bits": bits, "group_size": group_size} if bits == 4 else {"bits": 8}
    with open(os.path.join(var_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2)
    save_file(q_weights, os.path.join(var_dir, "model.safetensors"))
    size_mb = os.path.getsize(os.path.join(var_dir, "model.safetensors")) / 1e6
    print(f"✅ Export {variant_name.upper()} prêt : {out_dir} ({size_mb:.1f} Mo)")
    return out_dir

DIR_INT4 = quantize_and_save("int4", bits=4, group_size=32)
DIR_INT8 = quantize_and_save("int8", bits=8)

# Test d'écoute des deux variantes (INT4 puis INT8)
ref_wav = os.path.join(DATA_DIR, "ref_narrateur.wav")
ref_txt = open(os.path.join(DATA_DIR, "ref_narrateur.txt"), encoding="utf-8").read().strip()
TEST_TEXT = "Il déambula dans les rues désertes et les jardins publics d'Agrillon, réfléchissant avec gravité aux propos de l'estropier et le maudissant."

for variant, folder in [("int4", DIR_INT4), ("int8", DIR_INT8)]:
    t = TTS.from_pretrained(folder, variant=variant, device="cuda")
    v = t.voice(ref_wav, ref_txt, language="fr")
    aud = t.generate(TEST_TEXT, v, language="fr", seed=0)
    out_wav = os.path.join(WORK_DIR, f"test_reverb_{variant}.wav")
    sf.write(out_wav, aud, t.sample_rate)
    print(f"\n🎧 Écoute Reverb {variant.upper()} :")
    ipd.display(ipd.Audio(out_wav))

if globals().get("AUTO_DISCONNECT", False):
    import time
    from google.colab import runtime
    time.sleep(10)
    runtime.unassign()